In [1]:
# Импорты, версии и общие настройки — всё в одном месте, в начале блокнота
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:                       # sklearn нужен только проектам с моделями (с главы 6)
    import sklearn
    from sklearn.dummy import DummyRegressor
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error
    from sklearn.model_selection import train_test_split
except ImportError:
    sklearn = None

RANDOM_STATE = 42          # одно число для всей случайности: разбиения, модели, выборки
# папка с данными: data/ рядом с блокнотом (свой проект) или ../data (блокнот в папке курса)
DATA = next((p for p in (Path('data'), Path('../data')) if p.exists()), Path('data'))

pd.set_option('display.max_columns', 30)
print('Python', sys.version.split()[0], '| данные:', DATA.resolve())
for lib in (pd, np, matplotlib, sklearn):
    if lib is None:
        continue
    print(f'{lib.__name__:<11}{lib.__version__}')
from IPython.display import display, Markdown

Python 3.12.14 | данные: /private/tmp/course-learner-2026/data
pandas     3.0.6
numpy      2.4.6
matplotlib 3.10.9
sklearn    1.8.0


# Кто наши гости

## 1. Цель
Маркетингу нужны число анкет, портрет, активность и источники выручки; владельцу CRM — журнал и проблемы.

**Вывод:** отчёт пересчитывается на новых сырых анкетах и другом периоде заказов. Вне портфолио остаются исходные анкеты и очищенные персональные строки.

## 2. Данные

In [2]:
raw=pd.read_csv(DATA/"clients_raw.csv")
orders=pd.read_csv(DATA/"orders_3m.csv",parse_dates=["datetime"])
shops=pd.read_csv(DATA/"shops.csv")
print(raw.shape, orders.shape, shops.shape)
print(raw.dtypes)
print(raw.isna().sum())

(307, 9) (2089, 9) (7, 5)
client_id      str
name           str
gender         str
birth_date     str
city           str
phone          str
signup_date    str
bonus          str
source         str
dtype: object
client_id       0
name            0
gender         19
birth_date     19
city           16
phone          17
signup_date     0
bonus          15
source         27
dtype: int64


**Вывод:** анкеты и заказы имеют разное зерно. Анонимные заказы остаются в выручке сети, но не дают характеристик гостя.

## 3. Предобработка

In [3]:

SYNONYMS = {"соц. сети": "соцсети", "от друзей": "друзья", "проходил мимо": "мимо",
            "2гис": "карты", "яндекс карты": "карты"}

def clean_city(s):
    return (s.str.strip().str.lower().str.removeprefix("г.").str.strip()
             .replace({"екб": "екатеринбург"}).str.capitalize())

def clean_phone(s):
    digits = s.str.replace(r"\D", "", regex=True)
    ok = digits.str.fullmatch(r"[78]?\d{10}")      # 10 цифр или 11 с кодом 7/8
    return digits.where(ok).str[-10:]               # другие длины — пропуск, а не «похожий» номер

def clean_gender(s):
    return s.str.strip().str.lower().map({"м": "М", "муж": "М", "ж": "Ж", "жен": "Ж"})

def parse_birth(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    ru = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    born = iso.fillna(ru)
    return born.where((2026 - born.dt.year - (born.dt.strftime('%m%d') > '1201')).between(14, 100))

def clean_bonus(s):
    b = pd.to_numeric(s.str.replace(" ", ""), errors="coerce")
    return b.mask((b < 0) | (b > 10_000))

def clean_clients(raw):
    df = raw.drop_duplicates().copy()
    df["client_id"] = df["client_id"].str.strip().str.upper()
    df["name"] = df["name"].str.strip().str.capitalize()
    df["gender"] = clean_gender(df["gender"])
    df["birth_date"] = parse_birth(df["birth_date"])
    df["city"] = clean_city(df["city"])
    df["phone"] = clean_phone(df["phone"])
    df["signup_date"] = pd.to_datetime(df["signup_date"])
    df["bonus"] = clean_bonus(df["bonus"])
    df["source"] = df["source"].str.strip().str.lower().replace(SYNONYMS)
    return (df.sort_values("signup_date")
              .drop_duplicates(subset=["client_id"], keep="last")
              .reset_index(drop=True))

def report(raw, orders):
    clients=clean_clients(raw)
    o=orders.copy()
    o["client_id"]=o.client_id.str.strip().str.upper()
    g=o.groupby("client_id").agg(orders=("order_id","size"),revenue=("amount","sum"))
    guests=clients.merge(g,on="client_id",how="left",validate="one_to_one").fillna({"orders":0,"revenue":0})
    age=2026-clients.birth_date.dt.year-(clients.birth_date.dt.strftime("%m%d")>"1201")
    return {"clients":len(clients),"active_pct":round(100*guests.orders.gt(0).mean(),1),"median_age":age.median(),"avg_revenue_active":round(guests.loc[guests.orders>0,"revenue"].mean(),1)}


clients=clean_clients(raw)
log={"выгрузка":len(raw),"без полных дублей":len(raw.drop_duplicates()),"анкет":len(clients)}
assert clients.client_id.is_unique
age=2026-clients.birth_date.dt.year-(clients.birth_date.dt.strftime("%m%d")>"1201")
assert age.dropna().between(14,100).all()
print(log)
Path("private").mkdir(exist_ok=True)
clients.to_csv("private/clients_clean.csv",index=False)

{'выгрузка': 307, 'без полных дублей': 299, 'анкет': 293}


**Вывод:** сохранена последняя анкета каждой карты, удалены полные копии, недопустимые значения стали пропусками. Пол и телефон нормализуются функциями из видимой учебной заготовки.

## 3.1. Качество данных

In [4]:
missing_pct=(clients.isna().mean()*100).round(1).sort_values(ascending=False)
usable_age=round(100*clients.birth_date.notna().mean(),1)
print(missing_pct,usable_age)
tickets=[{"где":"CRM анкеты","правило":f"поле {col} неизвестно после очистки","сколько":int(clients[col].isna().sum()),"как найти":f"{col} is null"} for col in clients if clients[col].isna().any()]
print(tickets)

birth_date     14.3
source          9.2
bonus           7.8
gender          6.1
city            5.5
phone           5.5
client_id       0.0
name            0.0
signup_date     0.0
dtype: float64 85.7
[{'где': 'CRM анкеты', 'правило': 'поле gender неизвестно после очистки', 'сколько': 18, 'как найти': 'gender is null'}, {'где': 'CRM анкеты', 'правило': 'поле birth_date неизвестно после очистки', 'сколько': 42, 'как найти': 'birth_date is null'}, {'где': 'CRM анкеты', 'правило': 'поле city неизвестно после очистки', 'сколько': 16, 'как найти': 'city is null'}, {'где': 'CRM анкеты', 'правило': 'поле phone неизвестно после очистки', 'сколько': 16, 'как найти': 'phone is null'}, {'где': 'CRM анкеты', 'правило': 'поле bonus неизвестно после очистки', 'сколько': 23, 'как найти': 'bonus is null'}, {'где': 'CRM анкеты', 'правило': 'поле source неизвестно после очистки', 'сколько': 27, 'как найти': 'source is null'}]


**Вывод:** возраст и источники известны не у всех; эти пропуски нужно выяснять в CRM, а не заменять выдуманными значениями. Тикеты содержат агрегаты и условие поиска.

## 4.1. Портрет

In [5]:
clients["age"]=age
clients["age_group"]=pd.cut(age,[14,25,35,45,60,101],right=False,labels=["14–24","25–34","35–44","45–59","60–100"])
print(clients.age_group.value_counts(sort=False))
print(clients.city.value_counts())

age_group
14–24     38
25–34     61
35–44     60
45–59     84
60–100     8
Name: count, dtype: int64
city
Пермь           89
Казань          88
Тюмень          46
Уфа             24
Екатеринбург    22
Москва           8
Name: count, dtype: int64


**Вывод:** размеры возрастных групп и городов видны в таблицах; они относятся только к анкетам с известными признаками.

## 4.2. Анкеты и заказы

In [6]:
orders["client_id"]=orders.client_id.str.strip().str.upper()
g=orders.groupby("client_id").agg(orders=("order_id","size"),revenue=("amount","sum"))
guests=clients.merge(g,on="client_id",how="left",validate="one_to_one").fillna({"orders":0,"revenue":0})
active_pct=round(100*guests.orders.gt(0).mean(),1)
n_orphans=len(set(orders.client_id.dropna())-set(clients.client_id))
print(active_pct,n_orphans)

88.7 0


**Вывод:** анкеты без заказов сохранены, проверка карт-сирот отделена от анонимных заказов.

## 4.3. Источники

In [7]:
s=guests.source.fillna("не указано")
guests["source_group"]=s.where(s.map(s.value_counts())>=10,"прочее")
by_source=guests.assign(active=guests.orders.gt(0)).groupby("source_group").agg(clients=("client_id","size"),active_pct=("active","mean"),avg_revenue=("revenue","mean"))
by_source.active_pct=(100*by_source.active_pct).round(1)
by_source.avg_revenue=by_source.avg_revenue.round()
print(by_source.sort_values("clients",ascending=False))

              clients  active_pct  avg_revenue
source_group                                  
соцсети            80        87.5       1545.0
друзья             78        89.7       1431.0
мимо               41        87.8       1327.0
карты              32        87.5       1481.0
не указано         27        85.2       1523.0
реклама            27        92.6       1449.0
прочее              8       100.0       1340.0


**Вывод:** выручка здесь на зарегистрированного гостя, включая нулевую. Сравнение источников описательное, распределение по ним не было случайным.

## 4.4. Возраст и деньги

In [8]:
by_age=guests[guests.orders>0].groupby("age_group",observed=True).agg(guests_n=("client_id","size"),avg_orders=("orders","mean"),avg_revenue=("revenue","mean")).round(1)
print(by_age)

           guests_n  avg_orders  avg_revenue
age_group                                   
14–24            35         5.2       1744.6
25–34            54         4.5       1477.0
35–44            57         4.6       1701.3
45–59            70         4.9       1655.7
60–100            8         5.0       1681.9


**Вывод:** число гостей стоит рядом со средними; малые группы нельзя объявлять лучшими без оговорки о размере.

## 4.5. Свой город

In [9]:
t=orders.merge(shops[["shop_id","city"]],on="shop_id",how="left",validate="many_to_one").merge(clients[["client_id","city"]],on="client_id",how="left",suffixes=("_shop","_guest"),validate="many_to_one")
known=t.city_shop.notna() & t.city_guest.notna()
home_pct=round(100*t.loc[known,"city_shop"].eq(t.loc[known,"city_guest"]).mean(),1)
travelers=int((t.groupby("client_id").city_shop.nunique()>=2).sum())
print(home_pct,travelers)

85.1 73


**Вывод:** знаменатель доли включает только заказы с двумя известными городами, путешественник определяется по нескольким городам заказа.

## 5. Выводы и перенос

In [10]:
print("Осень",report(raw,orders))
print("Ноябрь",report(raw,orders[orders.datetime.dt.month==11]))

Осень {'clients': 293, 'active_pct': np.float64(88.7), 'median_age': np.float64(39.0), 'avg_revenue_active': np.float64(1646.0)}
Ноябрь {'clients': 293, 'active_pct': np.float64(73.0), 'median_age': np.float64(39.0), 'avg_revenue_active': np.float64(705.5)}


**Вывод:** очищены 307 строк до 293 анкет; активность осенью 88.7%, в ноябре 73.0%. Медианный возраст 39 лет не зависит от отбора заказов. Маркетингу предлагаем изучить источники с достаточной группой; владельцу CRM — заполнить причины пропусков. Эффект источника привлечения не измерен.